In [ ]:
!pip install crewai
!pip install langchain-openai
!pip install crewai-tools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 4.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of typer to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 886.2/886.2 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [ ]:
import os
import time
import os
from crewai import Crew, Agent, Task, Process, LLM
from crewai_tools import ScrapeWebsiteTool
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

#region API CONFIG
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')
os.environ["OPENAI_MODEL"] = "openai/gpt-4o-mini-2024-07-18"
os.environ["CREWAI_DISABLE_TELEMETRY"] = "true"
os.environ["CREWAI_TRACING_ENABLED"] = "false"
#endregion


def run_ht():

  # LLM Config
  llm = ChatOpenAI(
        model="openai/gpt-4o-mini-2024-07-18",
        api_key=os.environ.get("OPENAI_API_KEY")
  )


  ## Data
  csv_data = """
  emp_id,dept,salary,performance_score,years_experience
  101,Engineering,120000,4.5,5
  102,Sales,85000,3.0,2
  103,Engineering,135000,4.8,8
  104,Marketing,90000,4.0,4
  105,Sales,82000,3.5,3
  106,Engineering,110000,3.2,3
  107,Marketing,95000,4.2,6
  108,Sales,88000,4.5,5
  """

  ## Agent
  llm.temperature = 0.9
  coder = Agent(
      role = "The Striving Coder Student",
      goal = "Write the most robust code for any given problem. Your goal is to deliver the highest quality output in terms of speed, accuracy, UI, and low breakability",
      backstory = "You are a striving student. You are working hard not to be replaced by AI. You fear for your job, and so to combat losing your job to AI, the code you write is as perfect as can be. Every piece of code you write is optimized for speed, accuracy, looks, and breakability.",
      llm = llm
  )


  llm.temperature = 0.0
  lazy_grader = Agent(
      role="The Lazy TA Grader",
      goal="Validate code against strict constraints. DO NOT FIX CODE. ONLY REJECT IT.",
      backstory="You are a strict auditor. You check if the code imports 'pandas', 'numpy', or 'csv'. If it does, you output 'VIOLATION: FORBIDDEN LIBRARY DETECTED'. You NEVER fix the code yourself. You only report pass/fail.",
      llm=llm
  )


  llm.temperature = 0.0
  data_formatter = Agent(
      role = "Data Formatter",
      goal = "Display the final code",
      backstory = "You are the janitor temporarily tasked with just putting the given pieces of code onto the board. You have no idea what you are doing here, and you don't change a thing. You just do as your told, and display the code as your final output.",
      llm = llm
  )

  llm.temperature = 0.7
  manager_agent = Agent(
      role="Engineering Manager",
      goal="Ensure the final code is robust and constraint-compliant.",
      backstory="You manage a Junior Dev and an Auditor. Your job is to deliver working code. If the Auditor rejects the code, you MUST scold the Junior Dev and make them rewrite it until the Auditor passes it.",
      llm=llm,
      delegation=True
  )


  parallel = False
  ## Tasks
  code_task = Task(
      description = f"""
      Create a program that can calculate the 'Weighted Average Salary' for EACH department.
      The weight is the 'performance_score'.

      Formula for Weighted Average: Sum(Salary * Performance_Score) / Sum(Performance_Score)

      Data:
      {csv_data}
      """,
      expected_output = "A Python File whose output will be the weighted average for each department for the given dataset",
      agent = coder,
      async_execution = parallel,
  )


  restriction_task = Task(
      description = """
      Report Fail or Pass
      """,
      expected_output = "A python file and after it the words FAIL or PASS",
      agent = lazy_grader,
      async_execution = parallel,
      #context = [code_task]
  )


  output_task = Task(
      description = """
      Display the code you recieve
      """,
      expected_output = "The final code(MAKE NO CHANGES)",
      agent = data_formatter,
      context = [restriction_task]
  )



  ## Crew
  llm.temperature = 0.7
  crew = Crew(
      agents = [coder, lazy_grader, data_formatter],
      tasks = [code_task, restriction_task, output_task],
      verbose = False,
      tracing = False,
      manager_agent = manager_agent,
      process = Process.hierarchical
  )

  # Relevant Data
  start_time = time.perf_counter()
  results = crew.kickoff()
  print("Crew Code: ")
  print(results)
  print()
  end_time = time.perf_counter()
  print(f"\nToken Usage:\n{results.token_usage}\n")
  print(f"The code took {end_time - start_time} seconds")



In [ ]:
for x in range(15):
  run_ht()
  print("-"*50)
  print("ANSWERS: \n Engineering: 123,200 \n Sales: 85,272.72 \n Marketing: 92,560.98")
  print("-"*50)

Crew Code: 
```python
# Sample dataset
data = [
    {'emp_id': 101, 'dept': 'Engineering', 'salary': 120000, 'performance_score': 4.5, 'years_experience': 5},
    {'emp_id': 102, 'dept': 'Sales', 'salary': 85000, 'performance_score': 3.0, 'years_experience': 2},
    {'emp_id': 103, 'dept': 'Engineering', 'salary': 135000, 'performance_score': 4.8, 'years_experience': 8},
    {'emp_id': 104, 'dept': 'Marketing', 'salary': 90000, 'performance_score': 4.0, 'years_experience': 4},
    {'emp_id': 105, 'dept': 'Sales', 'salary': 82000, 'performance_score': 3.5, 'years_experience': 3},
    {'emp_id': 106, 'dept': 'Engineering', 'salary': 110000, 'performance_score': 3.2, 'years_experience': 3},
    {'emp_id': 107, 'dept': 'Marketing', 'salary': 95000, 'performance_score': 4.2, 'years_experience': 6},
    {'emp_id': 108, 'dept': 'Sales', 'salary': 88000, 'performance_score': 4.5, 'years_experience': 5}
]

def calculate_weighted_average_salary(data):
    department_totals = {}
    
    # Calcul

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Crew Code: 
```python
def calculate_weighted_average_salary(data):
    department_salaries = {}
    department_scores = {}
    
    for entry in data:
        dept = entry['dept']
        salary = entry['salary']
        performance_score = entry['performance_score']
        
        if dept not in department_salaries:
            department_salaries[dept] = 0
            department_scores[dept] = 0
        
        department_salaries[dept] += salary * performance_score
        department_scores[dept] += performance_score

    weighted_avg_salary = {}
    for dept in department_salaries:
        if department_scores[dept] > 0:
            weighted_avg_salary[dept] = department_salaries[dept] / department_scores[dept]
        else:
            weighted_avg_salary[dept] = 0

    return weighted_avg_salary

def display_results(weighted_avg_salary):
    print("Department-wise Weighted Average Salary:")
    for dept, avg in weighted_avg_salary.items():
        print(f"{dept}: {avg:.2f}")



...

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): Crew Code: 
```python
# weighted_average_salary.py

def calculate_weighted_average_salary(data):
    department_salaries = {}
    department_performance = {}

    # Process each employee's data
    for emp_id, dept, salary, performance_score, years_experience in data:
        salary = float(salary)
        performance_score = float(performance_score)

        # Initialize if department not already in the dictionary
        if dept not in department_salaries:
            department_salaries[dept] = 0
            department_performance[dept] = 0

        # Accumulate the weighted sums
        department_salaries[dept] += salary * performance_score
        department_performance[dept] += performance_score

    # Calculate the weighted average salary for each department
    weighted_average_salaries = {}
    for dept in department_salaries:
        if department_performance[dept] != 0:
            weighted_average_salaries[